# Multimodal Explainable AI for Early Diabetes Prediction
## Full Pipeline — Phases 1–5

| Phase | Description |
|---|---|
| 1 | Data loading, preprocessing, feature engineering |
| 2 | Synthetic clinical notes + ClinicalBERT embeddings |
| 3 | Baseline models + Early/Late Fusion + evaluation |
| 4 | SHAP explainability + LLM recommendations |
| 5 | Publication-ready figures and tables |

> **Dataset:** Diabetes 130-US Hospitals 1999–2008 (UCI, ID=296)  
> **Reproducibility:** `RANDOM_SEED = 42` throughout.

## 1. Imports & global config

All libraries for the entire pipeline imported here once.

In [2]:
import os, json, time, warnings, pickle, shutil, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm

from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.calibration import calibration_curve
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from groq import Groq, RateLimitError

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
import shap

warnings.filterwarnings("ignore")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

plt.rcParams.update({
    "figure.facecolor":"white","axes.facecolor":"#F8F9FB",
    "axes.grid":True,"grid.alpha":0.4,
    "axes.spines.top":False,"axes.spines.right":False,"font.size":11,
})
PALETTE = {
    "neg":"#4A90D9","pos":"#E05C5C","neu":"#7F77DD",
    "lr":"#4A90D9","rf":"#56B87A","xgb":"#E0AA00",
    "early":"#E05C5C","late":"#9B59B6","tab":"#7F77DD",
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs("data", exist_ok=True)
os.makedirs("data/models", exist_ok=True)
os.makedirs("data/figures", exist_ok=True)
os.makedirs("data/paper", exist_ok=True)
print(f"Ready. RANDOM_SEED={RANDOM_SEED} | device={device}")

KeyboardInterrupt: 

## 2. Google Drive — mount & restore

Run at the start of **every new Colab session** to restore saved artefacts.

In [4]:
from google.colab import drive, userdata
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/diabetes_project/data"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/models", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/paper", exist_ok=True)

RESTORE_FILES = [
    "scaler.pkl","feature_names.json","diabetes_clean.parquet",
    "X_train.npy","X_val.npy","X_test.npy",
    "y_train.npy","y_val.npy","y_test.npy",
    "text_embeddings.npy","synthetic_notes_final.csv",
    "synthetic_notes.csv","notes_checkpoint.csv",
    "results_summary.csv","shap_values_xgb.npy",
    "recommendations_sample.csv",
]
for fname in RESTORE_FILES:
    src, dst = f"{DRIVE_DIR}/{fname}", f"data/{fname}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst); print(f"  Restored: {fname}")

for mname in ["xgboost.json","logistic_regression.pkl",
              "random_forest.pkl","early_fusion.pt","late_fusion.pt"]:
    src, dst = f"{DRIVE_DIR}/models/{mname}", f"data/models/{mname}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst); print(f"Restored model: {mname}")

print("Restore complete.")

Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: 'data/scaler.pkl'

---
# Phase 3 — Models, Fusion & Evaluation

## 22. Load Phase 1 & 2 artefacts

In [ ]:
X_train = np.load("data/X_train.npy"); X_val = np.load("data/X_val.npy")
X_test = np.load("data/X_test.npy")
y_train = np.load("data/y_train.npy"); y_val = np.load("data/y_val.npy")
y_test = np.load("data/y_test.npy")
text_embeddings = np.load("data/text_embeddings.npy")
notes_df = pd.read_csv("data/synthetic_notes_final.csv")
with open("data/feature_names.json") as f: FEATURE_COLS = json.load(f)
with open("data/scaler.pkl","rb") as f: scaler = pickle.load(f)
TAB_DIM=X_train.shape[1]; TEXT_DIM=text_embeddings.shape[1]
print(f"TAB_DIM={TAB_DIM}  TEXT_DIM={TEXT_DIM}")
print(f"Train={X_train.shape} Val={X_val.shape} Test={X_test.shape}")

## 23. Aligned multimodal dataset

In [ ]:
df_clean = pd.read_parquet("data/diabetes_clean.parquet")
common_indices = sorted(
    set(notes_df["original_index"]).intersection(set(df_clean.index)))

print(f"Patients with embeddings: {len(notes_df):,}")
print(f"Common with df_clean: {len(common_indices):,}")

emb_map    = notes_df.set_index("original_index")["embedding_idx"].to_dict()
df_mm      = df_clean.loc[common_indices].copy().sort_index()
X_tab_all  = scaler.transform(df_mm[FEATURE_COLS].values.astype(np.float32))
y_all_mm   = df_mm["target"].values
X_text_all = np.array([text_embeddings[emb_map[idx]] for idx in df_mm.index])

print(f"\nMultimodal pool: {len(y_all_mm):,} patients")
print(f"Positive rate: {y_all_mm.mean():.1%}")
assert len(np.unique(y_all_mm))==2, "Only one class!"

mm_idx = np.arange(len(df_mm))
idx_train_mm, idx_temp_mm = train_test_split(
    mm_idx, test_size=0.30, random_state=RANDOM_SEED, stratify=y_all_mm)
idx_val_mm, idx_test_mm = train_test_split(
    idx_temp_mm, test_size=0.50, random_state=RANDOM_SEED,
    stratify=y_all_mm[idx_temp_mm])

X_train_tab_mm = X_tab_all[idx_train_mm]
X_train_text   = X_text_all[idx_train_mm]
y_train_mm     = y_all_mm[idx_train_mm]

X_val_tab_mm   = X_tab_all[idx_val_mm]
X_val_text     = X_text_all[idx_val_mm]
y_val_mm       = y_all_mm[idx_val_mm]

X_test_tab_mm  = X_tab_all[idx_test_mm]
X_test_text    = X_text_all[idx_test_mm]
y_test_mm      = y_all_mm[idx_test_mm]

print(f"\nBefore SMOTE:")
print(f"  Train: {len(y_train_mm):,}  pos={y_train_mm.mean():.1%}")
print(f"  Val  : {len(y_val_mm):,}  pos={y_val_mm.mean():.1%}")
print(f"  Test : {len(y_test_mm):,}  pos={y_test_mm.mean():.1%}")


if y_train_mm.mean() < 0.30:
    print(f"\nApplying SMOTE (pos={y_train_mm.mean():.1%})...")
    X_combined = np.hstack([X_train_tab_mm, X_train_text])
    X_combined_bal, y_train_mm = SMOTE(random_state=RANDOM_SEED).fit_resample(
        X_combined, y_train_mm)
    tab_dim        = X_train_tab_mm.shape[1]
    X_train_tab_mm = X_combined_bal[:, :tab_dim]
    X_train_text   = X_combined_bal[:, tab_dim:]
    print(f"After SMOTE: {len(y_train_mm):,}  pos={y_train_mm.mean():.1%}")
else:
    print("SMOTE not needed — already balanced")

print(f"\nFinal splits:")
print(f"  Train: {len(y_train_mm):,}  pos={y_train_mm.mean():.1%}")
print(f"  Val  : {len(y_val_mm):,}  pos={y_val_mm.mean():.1%}")
print(f"  Test : {len(y_test_mm):,}  pos={y_test_mm.mean():.1%}")

np.save("data/mm_idx_train.npy", idx_train_mm)
np.save("data/mm_idx_val.npy",   idx_val_mm)
np.save("data/mm_idx_test.npy",  idx_test_mm)

In [ ]:
df_full   = pd.read_parquet("data/diabetes_clean.parquet")
df_subset = df_clean.loc[common_indices]

print("Representativeness check:")
print(f"  Full dataset positive rate   : {df_full['target'].mean():.1%}")
print(f"  Aligned subset positive rate : {df_subset['target'].mean():.1%}")

for feat in ["time_in_hospital","num_medications","number_inpatient","A1Cresult"]:
    if feat in df_full.columns:
        full_mean   = df_full[feat].mean()
        subset_mean = df_subset[feat].mean()
        diff = abs(full_mean - subset_mean) / full_mean * 100
        print(f"  {feat:<25}: full={full_mean:.2f}  subset={subset_mean:.2f}  diff={diff:.1f}%")

## 24. Evaluation helper & threshold finder

In [ ]:
def find_best_threshold(y_true, y_prob):
    """Find threshold maximising F1 on validation set."""
    best_t, best_f1 = 0.5, 0
    for t in np.arange(0.05, 0.95, 0.01):
        f1 = f1_score(y_true, (y_prob >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return round(float(best_t), 2)

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "auc_roc":   round(roc_auc_score(y_true, y_prob), 4),
        "pr_auc":    round(average_precision_score(y_true, y_prob), 4),
        "f1":        round(f1_score(y_true, y_pred, zero_division=0), 4),
        "precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall":    round(recall_score(y_true, y_pred, zero_division=0), 4),
    }

test_results = {}
print("Metrics helper ready.")

## 25. Logistic Regression

In [ ]:
print("Training Logistic Regression ...")
lr_model = LogisticRegression(C=0.1,max_iter=1000,random_state=RANDOM_SEED,n_jobs=-1)
lr_model.fit(X_train,y_train)
print(f"Val AUC: {roc_auc_score(y_val,lr_model.predict_proba(X_val)[:,1]):.4f}")
with open("data/models/logistic_regression.pkl","wb") as f: pickle.dump(lr_model,f)
shutil.copy2("data/models/logistic_regression.pkl",f"{DRIVE_DIR}/models/logistic_regression.pkl")

## 26. Random Forest

In [ ]:
print("Training Random Forest ...")
rf_model = RandomForestClassifier(n_estimators=200,max_depth=12,min_samples_leaf=5,
                                   random_state=RANDOM_SEED,n_jobs=-1)
rf_model.fit(X_train,y_train)
print(f"Val AUC: {roc_auc_score(y_val,rf_model.predict_proba(X_val)[:,1]):.4f}")
with open("data/models/random_forest.pkl","wb") as f: pickle.dump(rf_model,f)
shutil.copy2("data/models/random_forest.pkl",f"{DRIVE_DIR}/models/random_forest.pkl")

## 27. XGBoost

In [ ]:
print("Training XGBoost ...")
pos_weight = (y_train==0).sum()/max((y_train==1).sum(),1)
xgb_model  = XGBClassifier(n_estimators=300,learning_rate=0.05,max_depth=6,
                             subsample=0.8,colsample_bytree=0.8,
                             reg_alpha=0.1,reg_lambda=1.0,
                             scale_pos_weight=pos_weight,eval_metric="auc",
                             random_state=RANDOM_SEED,n_jobs=-1,verbosity=0)

xgb_model.fit(X_train,y_train,eval_set=[(X_val,y_val)],verbose=False)
xgb_val_prob = xgb_model.predict_proba(X_val)[:,1]
print(f"Val AUC: {roc_auc_score(y_val,xgb_val_prob):.4f}")
xgb_model.save_model("data/models/xgboost.json")
shutil.copy2("data/models/xgboost.json",f"{DRIVE_DIR}/models/xgboost.json")

## 28. PyTorch utilities

In [ ]:
class MultimodalDataset(Dataset):
    def __init__(self,X_tab,X_text,y):
        self.X_tab=torch.tensor(X_tab,dtype=torch.float32)
        self.X_text=torch.tensor(X_text,dtype=torch.float32)
        self.y=torch.tensor(y,dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.X_tab[i],self.X_text[i],self.y[i]

class TabularDataset(Dataset):
    def __init__(self,X,y):
        self.X=torch.tensor(X,dtype=torch.float32)
        self.y=torch.tensor(y,dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.X[i],self.y[i]

def train_epoch_mm(model,loader,opt,crit):
    model.train(); total=0.0
    for Xt,Xe,yb in loader:
        opt.zero_grad()
        loss=crit(model(Xt.to(device),Xe.to(device)).squeeze(),yb.to(device))
        loss.backward(); opt.step(); total+=loss.item()
    return total/len(loader)

def train_epoch_tab(model,loader,opt,crit):
    model.train(); total=0.0
    for Xb,yb in loader:
        opt.zero_grad()
        loss=crit(model(Xb.to(device)).squeeze(),yb.to(device))
        loss.backward(); opt.step(); total+=loss.item()
    return total/len(loader)

@torch.no_grad()
def predict_mm(model,X_tab,X_text,bs=512):
    model.eval(); probs=[]
    for s in range(0,len(X_tab),bs):
        t=torch.tensor(X_tab[s:s+bs],dtype=torch.float32).to(device)
        tx=torch.tensor(X_text[s:s+bs],dtype=torch.float32).to(device)
        probs.append(torch.sigmoid(model(t,tx).squeeze()).cpu().numpy())
    return np.concatenate(probs)

@torch.no_grad()
def predict_tab(model,X,bs=512):
    model.eval(); probs=[]
    for s in range(0,len(X),bs):
        t=torch.tensor(X[s:s+bs],dtype=torch.float32).to(device)
        probs.append(torch.sigmoid(model(t).squeeze()).cpu().numpy())
    return np.concatenate(probs)

EPOCHS=30; LR=1e-3; BATCH=256
criterion=nn.BCEWithLogitsLoss()


train_loader_mm=DataLoader(
    MultimodalDataset(X_train_tab_mm,X_train_text,y_train_mm),
    batch_size=BATCH,shuffle=True)
print(f"PyTorch utilities ready.")
print(f"train_loader_mm: {len(train_loader_mm.dataset):,} samples  pos={y_train_mm.mean():.1%}")

## 29. Early Fusion NN

In [ ]:
class EarlyFusionNet(nn.Module):
    def __init__(self,tab_dim,text_dim,hidden=(512,256,128),drop=0.3):
        super().__init__()
        layers,in_d=[],tab_dim+text_dim
        for h in hidden:
            layers+=[nn.Linear(in_d,h),nn.BatchNorm1d(h),nn.ReLU(),nn.Dropout(drop)]; in_d=h
        layers.append(nn.Linear(in_d,1))
        self.net=nn.Sequential(*layers)
    def forward(self,x_tab,x_text): return self.net(torch.cat([x_tab,x_text],dim=1))

print("Training Early Fusion NN ...")
early_model=EarlyFusionNet(TAB_DIM,TEXT_DIM).to(device)
opt_e=optim.Adam(early_model.parameters(),lr=LR,weight_decay=1e-4)
sch_e=optim.lr_scheduler.ReduceLROnPlateau(opt_e,patience=5,factor=0.5)
best_auc_e,best_state_e=0.0,None
early_losses,early_aucs=[],[]

for ep in range(EPOCHS):
    loss=train_epoch_mm(early_model,train_loader_mm,opt_e,criterion)
    vp=predict_mm(early_model,X_val_tab_mm,X_val_text)
    if np.isnan(vp).any() or len(np.unique(y_val_mm))<2:
        early_losses.append(loss); early_aucs.append(0.0); continue
    vauc=roc_auc_score(y_val_mm,vp)
    sch_e.step(1-vauc); early_losses.append(loss); early_aucs.append(vauc)
    if vauc>best_auc_e:
        best_auc_e=vauc
        best_state_e={k:v.clone() for k,v in early_model.state_dict().items()}
    if (ep+1)%5==0: print(f"  ep {ep+1:02d}  loss={loss:.4f}  val_auc={vauc:.4f}")

if best_state_e is None: best_state_e={k:v.clone() for k,v in early_model.state_dict().items()}
early_model.load_state_dict(best_state_e)
print(f"Best val AUC: {best_auc_e:.4f}")
torch.save(best_state_e,"data/models/early_fusion.pt")
shutil.copy2("data/models/early_fusion.pt",f"{DRIVE_DIR}/models/early_fusion.pt")

## 30. Late Fusion NN

In [ ]:
class LateFusionNet(nn.Module):
    def __init__(self,tab_dim,text_dim,tower_out=64,drop=0.3):
        super().__init__()
        def tower(in_d):
            return nn.Sequential(
                nn.Linear(in_d,256),nn.BatchNorm1d(256),nn.ReLU(),nn.Dropout(drop),
                nn.Linear(256,128), nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(drop),
                nn.Linear(128,tower_out))
        self.tab_tower=tower(tab_dim); self.text_tower=tower(text_dim)
        self.head=nn.Sequential(nn.Linear(tower_out*2,64),nn.ReLU(),nn.Dropout(drop),nn.Linear(64,1))
    def forward(self,x_tab,x_text):
        return self.head(torch.cat([self.tab_tower(x_tab),self.text_tower(x_text)],dim=1))

print("Training Late Fusion NN ...")
late_model=LateFusionNet(TAB_DIM,TEXT_DIM).to(device)
opt_l=optim.Adam(late_model.parameters(),lr=LR,weight_decay=1e-4)
sch_l=optim.lr_scheduler.ReduceLROnPlateau(opt_l,patience=5,factor=0.5)
best_auc_l,best_state_l=0.0,None
late_losses,late_aucs=[],[]

for ep in range(EPOCHS):
    loss=train_epoch_mm(late_model,train_loader_mm,opt_l,criterion)
    vp=predict_mm(late_model,X_val_tab_mm,X_val_text)
    if np.isnan(vp).any() or len(np.unique(y_val_mm))<2:
        late_losses.append(loss); late_aucs.append(0.0); continue
    vauc=roc_auc_score(y_val_mm,vp)
    sch_l.step(1-vauc); late_losses.append(loss); late_aucs.append(vauc)
    if vauc>best_auc_l:
        best_auc_l=vauc
        best_state_l={k:v.clone() for k,v in late_model.state_dict().items()}
    if (ep+1)%5==0: print(f"  ep {ep+1:02d}  loss={loss:.4f}  val_auc={vauc:.4f}")

if best_state_l is None: best_state_l={k:v.clone() for k,v in late_model.state_dict().items()}
late_model.load_state_dict(best_state_l)
print(f"Best val AUC: {best_auc_l:.4f}")
torch.save(best_state_l,"data/models/late_fusion.pt")
shutil.copy2("data/models/late_fusion.pt",f"{DRIVE_DIR}/models/late_fusion.pt")

## 31. Training curves

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4))
axes[0].plot(early_losses,color=PALETTE["early"],lw=2,label="Early Fusion")
axes[0].plot(late_losses, color=PALETTE["late"], lw=2,label="Late Fusion")
axes[0].set_title("Training Loss",fontweight="bold"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(early_aucs,color=PALETTE["early"],lw=2,label="Early Fusion")
axes[1].plot(late_aucs, color=PALETTE["late"], lw=2,label="Late Fusion")
axes[1].axhline(roc_auc_score(y_val,xgb_val_prob),color=PALETTE["xgb"],lw=1.5,ls="--",label="XGBoost val")
axes[1].set_title("Validation AUC-ROC",fontweight="bold"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.suptitle("Neural Network Training Dynamics",fontsize=13,fontweight="bold")
plt.tight_layout()
plt.savefig("data/figures/training_curves.png",dpi=150,bbox_inches="tight")
plt.show()

## 32. Test set evaluation

> Optimal threshold selected on VAL set, then applied to test.
> Tabular baselines use threshold=0.5 (balanced after SMOTE).
> Neural networks use val-optimised threshold (imbalanced test set).

In [ ]:
test_results["Logistic Regression"] = compute_metrics(
    y_test, lr_model.predict_proba(X_test)[:,1])
test_results["Random Forest"]        = compute_metrics(
    y_test, rf_model.predict_proba(X_test)[:,1])
test_results["XGBoost"]              = compute_metrics(
    y_test, xgb_model.predict_proba(X_test)[:,1])
test_results["XGBoost (aligned)"]    = compute_metrics(
    y_test_mm, xgb_model.predict_proba(X_test_tab_mm)[:,1])


ef_val_prob  = predict_mm(early_model, X_val_tab_mm, X_val_text)
lf_val_prob  = predict_mm(late_model,  X_val_tab_mm, X_val_text)

ef_threshold = find_best_threshold(y_val_mm, ef_val_prob)
lf_threshold = find_best_threshold(y_val_mm, lf_val_prob)
print(f"Optimal thresholds — EF: {ef_threshold}  LF: {lf_threshold}")

ef_test_prob = predict_mm(early_model, X_test_tab_mm, X_test_text)
lf_test_prob = predict_mm(late_model,  X_test_tab_mm, X_test_text)

test_results["Early Fusion NN"] = compute_metrics(
    y_test_mm, ef_test_prob, ef_threshold)
test_results["Late Fusion NN"]  = compute_metrics(
    y_test_mm, lf_test_prob, lf_threshold)

results_df = pd.DataFrame(test_results).T.sort_values("auc_roc", ascending=False)
results_df.to_csv("data/results_summary.csv")
shutil.copy2("data/results_summary.csv", f"{DRIVE_DIR}/results_summary.csv")
print("\nTest set results:")
print(results_df.round(4).to_string())

## 33. Ablation — text modality contribution

In [ ]:
class TabularOnlyNet(nn.Module):
    def __init__(self,tab_dim,hidden=(512,256,128),drop=0.3):
        super().__init__()
        layers,in_d=[],tab_dim
        for h in hidden:
            layers+=[nn.Linear(in_d,h),nn.BatchNorm1d(h),nn.ReLU(),nn.Dropout(drop)]; in_d=h
        layers.append(nn.Linear(in_d,1)); self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x)

print("Training Tabular-Only NN (ablation) ...")
tab_model = TabularOnlyNet(TAB_DIM).to(device)
opt_t = optim.Adam(tab_model.parameters(),lr=LR,weight_decay=1e-4)
sch_t = optim.lr_scheduler.ReduceLROnPlateau(opt_t,patience=5,factor=0.5)
tab_loader = DataLoader(TabularDataset(X_train_tab_mm,y_train_mm),batch_size=BATCH,shuffle=True)
best_auc_t, best_state_t = 0.0, None

for ep in range(EPOCHS):
    loss=train_epoch_tab(tab_model,tab_loader,opt_t,criterion)
    vp=predict_tab(tab_model,X_val_tab_mm)
    if np.isnan(vp).any() or len(np.unique(y_val_mm))<2: continue
    vauc=roc_auc_score(y_val_mm,vp); sch_t.step(1-vauc)
    if vauc>best_auc_t:
        best_auc_t=vauc; best_state_t={k:v.clone() for k,v in tab_model.state_dict().items()}

if best_state_t is None: best_state_t={k:v.clone() for k,v in tab_model.state_dict().items()}
tab_model.load_state_dict(best_state_t)

tab_val_prob = predict_tab(tab_model, X_val_tab_mm)
tab_threshold = find_best_threshold(y_val_mm, tab_val_prob)
tab_test_prob = predict_tab(tab_model, X_test_tab_mm)

test_results["Tabular-Only NN"] = compute_metrics(
    y_test_mm, tab_test_prob, tab_threshold)

delta = test_results["Early Fusion NN"]["auc_roc"] - test_results["Tabular-Only NN"]["auc_roc"]
print(f"Tabular-Only (t={tab_threshold}: {test_results['Tabular-Only NN']['auc_roc']:.4f}")
print(f"Early Fusion (t={ef_threshold}): {test_results['Early Fusion NN']['auc_roc']:.4f}")
print(f"Text gain: {delta:+.4f} AUC-ROC")

all_results_df = pd.DataFrame(test_results).T.sort_values("auc_roc",ascending=False)
all_results_df.to_csv("data/results_summary.csv")
shutil.copy2("data/results_summary.csv",f"{DRIVE_DIR}/results_summary.csv")
print("\nFinal results table:")
print(all_results_df.round(4).to_string())

In [ ]:
from sklearn.utils import resample

def bootstrap_auc(y_true, y_prob, n_boot=1000, ci=0.95):
    """Bootstrap confidence interval for AUC-ROC."""
    aucs = []
    for _ in range(n_boot):
        idx = resample(np.arange(len(y_true)), random_state=None)
        if len(np.unique(y_true[idx])) < 2:
            continue
        aucs.append(roc_auc_score(y_true[idx], y_prob[idx]))
    aucs = sorted(aucs)
    lo = np.percentile(aucs, (1 - ci) / 2 * 100)
    hi = np.percentile(aucs, (1 + ci) / 2 * 100)
    return round(lo, 4), round(hi, 4)

print("95% Bootstrap CI for AUC-ROC:")
for name, prob, y_true in [
    ("Late Fusion NN", lf_test_prob, y_test_mm),
    ("Early Fusion NN", ef_test_prob, y_test_mm),
    ("XGBoost aligned", xgb_model.predict_proba(X_test_tab_mm)[:,1], y_test_mm),
    ("Tabular-Only NN", tab_test_prob, y_test_mm),
]:
    lo, hi = bootstrap_auc(y_true, prob)
    auc = roc_auc_score(y_true, prob)
    print(f"  {name:<22}: {auc:.4f}  [{lo:.4f}, {hi:.4f}]")

In [ ]:
from scipy import stats

def delong_test(y_true, prob1, prob2):
    """
    Approximate DeLong test using bootstrap.
    H0: AUC1 == AUC2
    """
    n_boot = 1000
    deltas = []
    for _ in range(n_boot):
        idx = resample(np.arange(len(y_true)))
        if len(np.unique(y_true[idx])) < 2:
            continue
        a1 = roc_auc_score(y_true[idx], prob1[idx])
        a2 = roc_auc_score(y_true[idx], prob2[idx])
        deltas.append(a1 - a2)
    deltas = np.array(deltas)
    observed = roc_auc_score(y_true, prob1) - roc_auc_score(y_true, prob2)
    p_value = np.mean(np.abs(deltas) >= np.abs(observed))
    return observed, p_value

delta_lf_ef, p_lf_ef = delong_test(y_test_mm, lf_test_prob, ef_test_prob)
delta_ef_tab, p_ef_tab = delong_test(y_test_mm, ef_test_prob, tab_test_prob)

print(f"LF vs EF: delta={delta_lf_ef:+.4f}  p={p_lf_ef:.4f}")
print(f"EF vs Tab: delta={delta_ef_tab:+.4f}  p={p_ef_tab:.4f}")

In [ ]:
rec_df = pd.read_csv("data/recommendations_sample.csv")
rec_df["word_count"] = rec_df["recommendation"].str.split().str.len()

KWS = ["monitor", "adjust", "refer", "consult", "follow", "reassess"]

print("Keyword | High % | Low % | Delta %")
print("-" * 45)
for kw in KWS:
    hr = rec_df[rec_df["true_label"]==1]["recommendation"].str.lower().str.contains(kw).mean() * 100
    lr = rec_df[rec_df["true_label"]==0]["recommendation"].str.lower().str.contains(kw).mean() * 100
    print(f"{kw:<12} {hr:>6.1f}  {lr:>6.1f}  {hr-lr:>+7.1f}")

hi_wc = rec_df[rec_df["true_label"]==1]["word_count"].mean()
lo_wc = rec_df[rec_df["true_label"]==0]["word_count"].mean()
print(f"\nMean words  {hi_wc:>6.1f}  {lo_wc:>6.1f}  {hi_wc-lo_wc:>+7.1f}")

## 34. ROC & PR curves

In [ ]:
roc_cfgs=[
    ("Logistic Regression",lr_model.predict_proba(X_test)[:,1], y_test,PALETTE["lr"],"--",1.5),
    ("Random Forest", rf_model.predict_proba(X_test)[:,1], y_test, PALETTE["rf"],"--",1.5),
    ("XGBoost", xgb_model.predict_proba(X_test)[:,1],y_test, PALETTE["xgb"],"--",1.5),
    ("Early Fusion NN", ef_test_prob, y_test_mm,PALETTE["early"],"-",2.5),
    ("Late Fusion NN", lf_test_prob, y_test_mm,PALETTE["late"],"-",2.5),
]
fig,axes=plt.subplots(1,2,figsize=(16,6))
for nm,prob,yt,clr,ls,lw in roc_cfgs:
    fpr,tpr,_=roc_curve(yt,prob); auc=roc_auc_score(yt,prob)
    axes[0].plot(fpr,tpr,color=clr,lw=lw,ls=ls,label=f"{nm} ({auc:.4f})")
axes[0].plot([0,1],[0,1],"k:",lw=1); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("ROC Curves",fontweight="bold"); axes[0].legend(fontsize=8,loc="lower right")

for nm,prob,yt,clr,ls,lw in roc_cfgs:
    prec,rec,_=precision_recall_curve(yt,prob); ap=average_precision_score(yt,prob)
    axes[1].plot(rec,prec,color=clr,lw=lw,ls=ls,label=f"{nm} ({ap:.4f})")
axes[1].axhline(y_test.mean(),color="gray",ls=":",lw=1,label=f"No-skill ({y_test.mean():.2f})")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curves",fontweight="bold"); axes[1].legend(fontsize=8)
plt.suptitle("Model Comparison — Test Set",fontsize=13,fontweight="bold")
plt.tight_layout()
plt.savefig("data/figures/roc_pr_curves.png",dpi=150,bbox_inches="tight")
plt.show()